# Python File Handling: Working with CSV, JSON, and Pickle

This notebook provides a comprehensive guide to handling files in Python with a focus on three important formats:
- CSV (Comma Separated Values) - For tabular data
- JSON (JavaScript Object Notation) - For structured data exchange
- Pickle - For Python object serialization

We'll explore how to read, write, and manipulate data in each format, along with best practices and common pitfalls.

## Table of Contents
1. [Import Required Libraries](#libraries)
2. [Working with CSV Files](#csv)
3. [Working with JSON Files](#json)
4. [Working with Pickle Files](#pickle)
5. [Comparing File Formats](#comparison)
6. [Error Handling in File Operations](#error-handling)
7. [Practical Examples and Use Cases](#examples)

## 1. Import Required Libraries <a id="libraries"></a>

Let's start by importing all the libraries we'll need throughout this notebook:

In [ ]:
# Standard libraries for file operations
import csv
import json
import pickle
import os
from pathlib import Path

# Third-party libraries for enhanced functionality
import pandas as pd
import numpy as np

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# For creating demo data
from datetime import datetime
from pprint import pprint

# Check versions
print(f"Python version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Working with CSV Files <a id="csv"></a>

CSV (Comma-Separated Values) is a simple file format used to store tabular data. Each line in a CSV file represents a row in a table, and columns are separated by commas (or other delimiters).

### 2.1 Creating Sample CSV Data

First, let's create a sample CSV file to work with:

In [ ]:
# Create a directory for our sample files if it doesn't exist
sample_dir = Path("sample_data")
sample_dir.mkdir(exist_ok=True)

# Sample data - employee records
employee_data = [
    ["id", "name", "department", "salary", "hire_date"],
    [1, "John Smith", "Engineering", 85000, "2020-01-15"],
    [2, "Mary Johnson", "Marketing", 76000, "2019-03-20"],
    [3, "James Brown", "Engineering", 92000, "2018-11-10"],
    [4, "Patricia Davis", "HR", 65000, "2021-05-05"],
    [5, "Robert Wilson", "Marketing", 79000, "2020-08-12"]
]

# Write to CSV using the built-in csv module
csv_file_path = sample_dir / "employees.csv"

with open(csv_file_path, 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerows(employee_data)
    
print(f"Created CSV file at: {csv_file_path}")

# Let's view the first few lines of the file
with open(csv_file_path, 'r') as csvfile:
    content = [next(csvfile) for _ in range(3)]
    
print("\nFirst 3 lines of the CSV file:")
print(''.join(content))

### 2.2 Reading CSV Files Using the csv Module

Python's built-in `csv` module provides functionality to read and write CSV files:

In [ ]:
# Reading CSV file using csv.reader
with open(csv_file_path, 'r', newline='') as csvfile:
    csv_reader = csv.reader(csvfile)
    
    # Get the header
    header = next(csv_reader)
    print(f"CSV Header: {header}")
    
    # Read and print the rows
    print("\nEmployee Data:")
    for row in csv_reader:
        print(row)

### 2.3 Reading CSV Files Using DictReader

The `csv.DictReader` provides a dictionary-like interface for reading CSV files:

In [ ]:
# Reading CSV using DictReader (gives us named fields)
with open(csv_file_path, 'r', newline='') as csvfile:
    dict_reader = csv.DictReader(csvfile)
    
    print("Employee Data with Named Fields:")
    for row in dict_reader:
        print(f"ID: {row['id']}, Name: {row['name']}, Department: {row['department']}, Salary: ${row['salary']}")

### 2.4 Reading CSV Files Using pandas

Pandas provides a much more powerful and convenient way to work with CSV files:

In [ ]:
# Reading CSV with pandas
df_employees = pd.read_csv(csv_file_path)

# Display the DataFrame
print("Pandas DataFrame from CSV:")
display(df_employees)

# Basic information about the DataFrame
print("\nDataFrame Information:")
print(f"Shape: {df_employees.shape}")
print(f"Columns: {df_employees.columns.tolist()}")
print("\nSummary Statistics:")
display(df_employees.describe())

### 2.5 Writing CSV Files

Now let's see how to write data to CSV files using both the native `csv` module and pandas:

In [ ]:
# First, let's modify our data slightly (adding a new employee and a raise for everyone)
df_employees['salary'] = df_employees['salary'] * 1.1  # 10% raise for everyone

# Add a new employee
new_employee = pd.DataFrame({
    'id': [6],
    'name': ['Sarah Lee'],
    'department': ['Engineering'],
    'salary': [88000],
    'hire_date': ['2022-01-10']
})

df_updated = pd.concat([df_employees, new_employee], ignore_index=True)
display(df_updated)

# Writing with csv.writer
updated_csv_path = sample_dir / "updated_employees.csv"

with open(updated_csv_path, 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    
    # Write header
    csv_writer.writerow(df_updated.columns)
    
    # Write data
    for _, row in df_updated.iterrows():
        csv_writer.writerow(row)

print(f"Created updated CSV file at: {updated_csv_path}")

# Writing with pandas (much simpler!)
pandas_csv_path = sample_dir / "pandas_employees.csv"
df_updated.to_csv(pandas_csv_path, index=False)
print(f"Created pandas CSV file at: {pandas_csv_path}")

### 2.6 Working with Different Delimiters

CSV files don't always use commas as separators. Let's create and read a TSV (Tab-Separated Values) file:

In [ ]:
# Creating a TSV file
tsv_file_path = sample_dir / "employees.tsv"
df_updated.to_csv(tsv_file_path, sep='\t', index=False)

print(f"Created TSV file at: {tsv_file_path}")

# Reading a TSV file
tsv_data = pd.read_csv(tsv_file_path, sep='\t')
print("Data read from TSV file:")
display(tsv_data.head())

### 2.7 Working with Custom CSV Dialects

CSV has various dialects (different formats and conventions). The `csv` module allows you to define and use custom dialects:

In [ ]:
# Register a custom dialect
csv.register_dialect('custom', 
                    delimiter='|',
                    quoting=csv.QUOTE_ALL,
                    doublequote=True,
                    lineterminator='\n')

# Write using the custom dialect
custom_csv_path = sample_dir / "custom_employees.csv"

with open(custom_csv_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile, dialect='custom')
    writer.writerow(df_updated.columns)
    for _, row in df_updated.iterrows():
        writer.writerow(row)

print(f"Created custom CSV file at: {custom_csv_path}")

# Read the first few lines of the custom CSV
with open(custom_csv_path, 'r') as file:
    for _ in range(3):
        print(file.readline().strip())
        
# Reading with the custom dialect
with open(custom_csv_path, 'r', newline='') as csvfile:
    reader = csv.reader(csvfile, dialect='custom')
    header = next(reader)
    first_row = next(reader)
    print(f"\nHeader: {header}")
    print(f"First row: {first_row}")

## 3. Working with JSON Files <a id="json"></a>

JSON (JavaScript Object Notation) is a lightweight data interchange format that is easy for humans to read and write and easy for machines to parse and generate. It's widely used for configuration files, API responses, and data exchange.

### 3.1 Creating Sample JSON Data

In [ ]:
# Sample employee data as a Python dictionary
employee_dict = {
    "company": "Tech Innovations Inc.",
    "location": "San Francisco",
    "employees": [
        {
            "id": 1,
            "name": "John Smith",
            "department": "Engineering",
            "salary": 85000,
            "hire_date": "2020-01-15",
            "skills": ["Python", "JavaScript", "Docker"],
            "contact": {
                "email": "john.smith@example.com",
                "phone": "555-123-4567"
            }
        },
        {
            "id": 2,
            "name": "Mary Johnson",
            "department": "Marketing",
            "salary": 76000,
            "hire_date": "2019-03-20",
            "skills": ["SEO", "Content Strategy", "Social Media"],
            "contact": {
                "email": "mary.johnson@example.com",
                "phone": "555-234-5678"
            }
        },
        {
            "id": 3,
            "name": "James Brown",
            "department": "Engineering",
            "salary": 92000,
            "hire_date": "2018-11-10",
            "skills": ["Python", "Machine Learning", "AWS"],
            "contact": {
                "email": "james.brown@example.com",
                "phone": "555-345-6789"
            }
        }
    ],
    "founded": 2010,
    "active": True
}

# Print the dictionary
print("Python Dictionary:")
pprint(employee_dict, width=100)

### 3.2 Writing JSON Files

Let's write our Python dictionary to a JSON file:

In [ ]:
# Write JSON to a file
json_file_path = sample_dir / "employees.json"

with open(json_file_path, 'w') as jsonfile:
    json.dump(employee_dict, jsonfile)

print(f"Created JSON file at: {json_file_path}")

# Write JSON with pretty formatting (indentation)
pretty_json_path = sample_dir / "employees_pretty.json"

with open(pretty_json_path, 'w') as jsonfile:
    json.dump(employee_dict, jsonfile, indent=4, sort_keys=True)

print(f"Created pretty-printed JSON file at: {pretty_json_path}")

# Let's see the difference - read the first few lines of each file
print("\nRegular JSON (first 100 chars):")
with open(json_file_path, 'r') as file:
    print(file.read(100) + "...")

print("\nPretty JSON (first 200 chars):")
with open(pretty_json_path, 'r') as file:
    print(file.read(200) + "...")

### 3.3 Reading JSON Files

Now let's read the JSON file back into Python:

In [ ]:
# Read JSON from a file
with open(json_file_path, 'r') as jsonfile:
    loaded_data = json.load(jsonfile)

print("JSON loaded back into Python:")
print(f"Company: {loaded_data['company']}")
print(f"Number of employees: {len(loaded_data['employees'])}")
print(f"First employee name: {loaded_data['employees'][0]['name']}")
print(f"First employee skills: {', '.join(loaded_data['employees'][0]['skills'])}")

### 3.4 Working with JSON Strings

Sometimes JSON comes as a string (e.g., from an API response) rather than a file:

In [ ]:
# Convert Python dict to JSON string
json_str = json.dumps(employee_dict, indent=2)
print("JSON string representation:")
print(json_str[:500] + "...")  # Print first 500 chars

# Parse JSON string back to Python dict
parsed_dict = json.loads(json_str)
print("\nParsed back to Python dict:")
print(f"Company: {parsed_dict['company']}")
print(f"First employee email: {parsed_dict['employees'][0]['contact']['email']}")

### 3.5 Converting Between JSON and pandas

Pandas provides convenient functions to work with JSON data:

In [ ]:
# Convert the employees list to a DataFrame
df_from_json = pd.DataFrame(employee_dict["employees"])
print("DataFrame created from JSON data:")
display(df_from_json)

# Handle nested JSON data (flatten the contact info)
df_expanded = pd.json_normalize(employee_dict["employees"])
print("\nFlattened nested JSON data:")
display(df_expanded)

# Convert DataFrame back to JSON
json_from_df = df_expanded.to_json(orient="records")
print("\nJSON from DataFrame (first 200 chars):")
print(json_from_df[:200] + "...")

# Pretty print the JSON from DataFrame
json_pretty = df_expanded.to_json(orient="records", indent=2)
print("\nPretty JSON from DataFrame:")
print(json_pretty)

### 3.6 Handling JSON Errors

Let's look at some common errors when working with JSON:

In [ ]:
# Example 1: Invalid JSON string
invalid_json = '{"name": "John", "age": 30, "city": "New York"'  # Missing closing brace

try:
    parsed_json = json.loads(invalid_json)
except json.JSONDecodeError as e:
    print(f"JSON Decode Error: {e}")

# Example 2: Non-JSON serializable data
try:
    # Create a class that's not JSON serializable
    class Person:
        def __init__(self, name):
            self.name = name
    
    person = Person("Alice")
    json_data = json.dumps({"person": person})
except TypeError as e:
    print(f"Type Error: {e}")
    
    # Solution: Create a custom JSON encoder
    class CustomEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, Person):
                return {"name": obj.name}
            return super().default(obj)
    
    # Now it works
    json_data = json.dumps({"person": person}, cls=CustomEncoder)
    print(f"Custom encoded JSON: {json_data}")

## 4. Working with Pickle Files <a id="pickle"></a>

Pickle is a Python-specific data serialization module. It can serialize nearly any Python object directly to a binary file format. This is particularly useful for saving machine learning models or complex Python data structures.

### 4.1 Creating Sample Data for Pickling

In [ ]:
# Let's create some complex Python data structures to pickle
complex_data = {
    "name": "Complex Data Structure",
    "created_at": datetime.now(),
    "numeric_data": np.array([1, 2, 3, 4, 5]),
    "dataframe": pd.DataFrame({
        "A": np.random.rand(5),
        "B": np.random.rand(5)
    }),
    "nested_dict": {
        "level1": {
            "level2": {
                "level3": "Deep nesting"
            }
        }
    },
    "function": lambda x: x*2  # Note: lambdas can be pickled but it's not recommended
}

print("Complex data structure to be pickled:")
for key, value in complex_data.items():
    print(f"{key}: {type(value)}")
    if isinstance(value, (pd.DataFrame, np.ndarray)):
        print(value)

### 4.2 Pickling Python Objects

Let's pickle our complex data structure and save it to a file:

In [ ]:
# Save to a pickle file
pickle_file_path = sample_dir / "complex_data.pkl"

with open(pickle_file_path, 'wb') as pickle_file:  # Note: 'wb' for binary write
    pickle.dump(complex_data, pickle_file)
    
print(f"Created pickle file at: {pickle_file_path}")
print(f"File size: {os.path.getsize(pickle_file_path)} bytes")

### 4.3 Unpickling Python Objects

Now let's load the pickled data back into Python:

In [ ]:
# Load from a pickle file
with open(pickle_file_path, 'rb') as pickle_file:  # Note: 'rb' for binary read
    loaded_data = pickle.load(pickle_file)
    
print("Loaded data types:")
for key, value in loaded_data.items():
    print(f"{key}: {type(value)}")

print("\nLoaded DataFrame:")
display(loaded_data["dataframe"])

print("\nLoaded NumPy Array:")
print(loaded_data["numeric_data"])

print("\nOriginal creation timestamp:")
print(loaded_data["created_at"])

# Test the unpickled function
print("\nTesting unpickled function:")
print(f"function(5) = {loaded_data['function'](5)}")

### 4.4 Pickle Protocol Versions

Pickle has different protocol versions that affect compatibility and efficiency:

In [ ]:
# Check available pickle protocols
print(f"Highest available pickle protocol version: {pickle.HIGHEST_PROTOCOL}")

# Save with different protocol versions
for protocol in range(pickle.HIGHEST_PROTOCOL + 1):
    protocol_file = sample_dir / f"protocol_{protocol}.pkl"
    
    with open(protocol_file, 'wb') as f:
        pickle.dump(complex_data, f, protocol=protocol)
    
    file_size = os.path.getsize(protocol_file)
    print(f"Protocol {protocol} file size: {file_size} bytes")
    
    # Test loading
    with open(protocol_file, 'rb') as f:
        loaded = pickle.load(f)
    
    print(f"Successfully loaded from protocol {protocol}")

### 4.5 Security Considerations with Pickle

**Important**: Pickle is not secure against malicious data! Never unpickle data from untrusted sources!

In [ ]:
# WARNING: For educational purposes only
print("SECURITY WARNING:")
print("- Never unpickle data from untrusted sources")
print("- Malicious pickle data can execute arbitrary code")
print("- Use secure alternatives like JSON for untrusted data")

# Let's demonstrate a safer alternative for simple data:
safe_data = {
    "name": "Safe Data",
    "values": [1, 2, 3, 4, 5],
    "nested": {"key": "value"}
}

# JSON is safer for data from untrusted sources
json_safe_path = sample_dir / "safe_data.json"
with open(json_safe_path, 'w') as f:
    json.dump(safe_data, f)

print(f"\nCreated safe JSON file at: {json_safe_path}")

### 4.6 When to Use Pickle

Pickle is most appropriate for:
1. Storing Python-specific objects temporarily
2. Saving machine learning models
3. Inter-process communication within trusted Python environments
4. Caching complex Python objects

Let's see a typical use case with a machine learning model:

In [ ]:
# Example: Pickling a simple machine learning model
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# Generate a synthetic dataset
X, y = make_classification(n_samples=100, n_features=4, random_state=42)

# Create and train a model
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X, y)

# Pickle the trained model
model_path = sample_dir / "random_forest_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"Saved trained model to {model_path}")

# Later, load the model and make predictions
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Make predictions with the loaded model
predictions = loaded_model.predict(X[:5])
print(f"Predictions from loaded model: {predictions}")
print(f"Actual values: {y[:5]}")

## 5. Comparing File Formats <a id="comparison"></a>

Let's compare CSV, JSON, and Pickle formats across various dimensions:

In [ ]:
# Create a simple DataFrame for comparison
comparison_df = pd.DataFrame({
    'id': range(1, 6),
    'name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva'],
    'score': np.random.rand(5) * 100,
    'active': [True, False, True, True, False],
    'timestamp': pd.date_range(start='2023-01-01', periods=5)
})

print("Sample data for comparison:")
display(comparison_df)

# Save in all three formats
comp_csv_path = sample_dir / "comparison.csv"
comp_json_path = sample_dir / "comparison.json"
comp_pkl_path = sample_dir / "comparison.pkl"

# CSV
comparison_df.to_csv(comp_csv_path, index=False)

# JSON
comparison_df.to_json(comp_json_path, orient='records', date_format='iso')

# Pickle
comparison_df.to_pickle(comp_pkl_path)

# Compare file sizes
csv_size = os.path.getsize(comp_csv_path)
json_size = os.path.getsize(comp_json_path)
pkl_size = os.path.getsize(comp_pkl_path)

# Create a comparison DataFrame
format_comparison = pd.DataFrame({
    'Format': ['CSV', 'JSON', 'Pickle'],
    'File Size (bytes)': [csv_size, json_size, pkl_size],
    'Human Readable': ['Yes', 'Yes', 'No'],
    'Cross-language': ['Yes', 'Yes', 'No (Python only)'],
    'Data Types Preserved': ['No', 'Limited', 'Yes'],
    'Security': ['Safe', 'Safe', 'Unsafe for untrusted sources']
})

print("\nFormat Comparison:")
display(format_comparison)

# Plot the file sizes
plt.figure(figsize=(10, 6))
sns.barplot(x='Format', y='File Size (bytes)', data=format_comparison)
plt.title('File Size Comparison')
plt.ylabel('Size in bytes')
plt.grid(axis='y', alpha=0.3)
plt.show()

### 5.1 When to Use Each Format

**Use CSV when:**
- Working with tabular data
- Need human-readable files
- Interoperability with other tools and languages is important
- Data is simple (mostly strings and numbers)

**Use JSON when:**
- Working with hierarchical or nested data
- Need human-readable files
- Interoperability with web applications and APIs is important
- Need to preserve some data structure

**Use Pickle when:**
- Working with Python-specific objects
- Need to preserve complex Python objects (like functions, classes)
- Working in a trusted Python-only environment
- Need to save machine learning models or other complex objects
- Performance and maintaining exact object states are priorities

## 6. Error Handling in File Operations <a id="error-handling"></a>

Let's explore proper error handling for file operations:

In [ ]:
# Common file operation errors and how to handle them

# 1. FileNotFoundError
try:
    with open('non_existent_file.txt', 'r') as f:
        content = f.read()
except FileNotFoundError:
    print("Error: The file does not exist!")

# 2. PermissionError
try:
    # This would fail if the file exists and is read-only
    with open('/etc/passwd', 'w') as f:
        f.write('test')
except PermissionError:
    print("Error: You don't have permission to modify this file!")

# 3. IsADirectoryError
try:
    with open('/tmp', 'r') as f:
        content = f.read()
except IsADirectoryError:
    print("Error: This is a directory, not a file!")

# 4. UnicodeDecodeError
try:
    # This would fail if the file contains binary data
    with open(pickle_file_path, 'r') as f:  # Using text mode for a binary file
        content = f.read()
except UnicodeDecodeError:
    print("Error: This file contains binary data and can't be read as text!")

### 6.1 Using Context Managers for Safe File Operations

Context managers (`with` statement) automatically handle file closing, even if exceptions occur:

In [ ]:
# Bad practice - file might not get closed if an error occurs
def bad_file_read(filename):
    f = open(filename, 'r')
    content = f.read()
    # If an exception occurs here, the file might not be closed
    f.close()
    return content

# Good practice - with context manager
def good_file_read(filename):
    try:
        with open(filename, 'r') as f:
            content = f.read()
            return content
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found!")
        return None
    except Exception as e:
        print(f"Error reading file: {str(e)}")
        return None

# Test with a file that exists
print(good_file_read(str(csv_file_path)))

# Test with a file that doesn't exist
print(good_file_read("non_existent_file.txt"))

### 6.2 Creating a Robust File Processing Function

Let's create a robust function that can handle reading from any of our file types:

In [ ]:
def read_data_file(file_path, file_format=None):
    """
    Safely read data from CSV, JSON, or Pickle file
    
    Parameters:
    -----------
    file_path : str or Path
        Path to the file to read
    file_format : str, optional
        Format of the file ('csv', 'json', 'pickle'). If None, inferred from file extension
        
    Returns:
    --------
    data : object
        The data read from the file
    """
    file_path = Path(file_path)  # Convert to Path object
    
    # Determine format from file extension if not specified
    if file_format is None:
        file_format = file_path.suffix.lower()[1:]  # Remove the dot
    
    try:
        # CSV
        if file_format == 'csv':
            try:
                return pd.read_csv(file_path)
            except pd.errors.EmptyDataError:
                print("Warning: Empty CSV file!")
                return pd.DataFrame()
            except pd.errors.ParserError:
                print("Error: Malformed CSV file!")
                return None
        
        # JSON
        elif file_format == 'json':
            try:
                with open(file_path, 'r') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                print("Error: Malformed JSON file!")
                return None
        
        # Pickle
        elif file_format == 'pkl' or file_format == 'pickle':
            try:
                with open(file_path, 'rb') as f:
                    return pickle.load(f)
            except pickle.UnpicklingError:
                print("Error: Malformed Pickle file!")
                return None
        
        # Unknown format
        else:
            print(f"Error: Unsupported file format '{file_format}'")
            return None
            
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found!")
        return None
    except PermissionError:
        print(f"Error: No permission to read '{file_path}'!")
        return None
    except Exception as e:
        print(f"Unexpected error reading '{file_path}': {str(e)}")
        return None

# Test our function with different file types
print("Reading CSV file:")
csv_data = read_data_file(csv_file_path)
display(csv_data.head())

print("\nReading JSON file:")
json_data = read_data_file(json_file_path)
print(f"Company: {json_data['company']}")
print(f"Number of employees: {len(json_data['employees'])}")

print("\nReading Pickle file:")
pickle_data = read_data_file(pickle_file_path)
print(f"Creation timestamp: {pickle_data['created_at']}")

## 7. Practical Examples and Use Cases <a id="examples"></a>

Let's explore some real-world examples of working with these file formats.

### 7.1 Data Conversion Between Formats

In [ ]:
def convert_file_format(input_file, output_file):
    """Convert data between CSV, JSON, and Pickle formats"""
    input_path = Path(input_file)
    output_path = Path(output_file)
    
    # Read the input data
    data = read_data_file(input_path)
    if data is None:
        return False
    
    # Determine output format
    output_format = output_path.suffix.lower()[1:]
    
    try:
        # Convert to DataFrame if needed
        if not isinstance(data, pd.DataFrame):
            if isinstance(data, dict) and "employees" in data and isinstance(data["employees"], list):
                # Handle our specific JSON structure
                data = pd.DataFrame(data["employees"])
            else:
                # Generic conversion
                data = pd.DataFrame(data)
        
        # Write to the output format
        if output_format == 'csv':
            data.to_csv(output_path, index=False)
        elif output_format == 'json':
            data.to_json(output_path, orient='records', indent=2)
        elif output_format in ['pkl', 'pickle']:
            data.to_pickle(output_path)
        else:
            print(f"Error: Unsupported output format '{output_format}'")
            return False
            
        print(f"Successfully converted {input_path} to {output_path}")
        return True
        
    except Exception as e:
        print(f"Error converting file: {str(e)}")
        return False

# Test conversion from JSON to CSV
convert_file_format(json_file_path, sample_dir / "converted_from_json.csv")

# Test conversion from CSV to Pickle
convert_file_format(csv_file_path, sample_dir / "converted_from_csv.pkl")

# Verify the conversion results
print("\nData from converted JSON to CSV:")
display(pd.read_csv(sample_dir / "converted_from_json.csv"))

print("\nData from converted CSV to Pickle:")
display(pd.read_pickle(sample_dir / "converted_from_csv.pkl"))

### 7.2 Working with Configuration Files

JSON is commonly used for configuration files. Let's create a configuration system:

In [ ]:
# Create a default configuration
default_config = {
    "app_name": "My Data Processing App",
    "version": "1.0.0",
    "max_threads": 4,
    "log_level": "INFO",
    "data_dir": "./data",
    "output_dir": "./output",
    "file_formats": ["csv", "json", "xlsx"],
    "database": {
        "host": "localhost",
        "port": 5432,
        "username": "user",
        "password": "password",
        "db_name": "mydb"
    }
}

config_path = sample_dir / "config.json"

# Write default config to file
with open(config_path, 'w') as f:
    json.dump(default_config, f, indent=2)

print(f"Created config file at {config_path}")

# Function to load config
def load_config(config_path, default=None):
    try:
        with open(config_path, 'r') as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading config: {str(e)}")
        return default if default else {}

# Function to update config
def update_config(config_path, updates):
    config = load_config(config_path, {})
    
    # Update config recursively
    def update_dict(d, u):
        for k, v in u.items():
            if isinstance(v, dict) and k in d and isinstance(d[k], dict):
                update_dict(d[k], v)
            else:
                d[k] = v
    
    update_dict(config, updates)
    
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    return config

# Test loading config
app_config = load_config(config_path)
print("Loaded configuration:")
pprint(app_config)

# Test updating config
updated = update_config(config_path, {
    "log_level": "DEBUG",
    "database": {
        "port": 5433,
        "password": "new_password"
    },
    "new_setting": "value"
})

print("\nUpdated configuration:")
pprint(updated)

### 7.3 Working with Real-World Structured Data Files

Let's work with some more complex data operations:

In [ ]:
# Create a simulated sales dataset
np.random.seed(42)
n_records = 1000

products = ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard', 'Mouse', 'Headphones', 'Speaker']
regions = ['North', 'South', 'East', 'West', 'Central']
channels = ['Online', 'Retail', 'Direct', 'Wholesale']

sales_data = pd.DataFrame({
    'date': pd.date_range(start='2022-01-01', periods=n_records),
    'product': np.random.choice(products, n_records),
    'region': np.random.choice(regions, n_records),
    'channel': np.random.choice(channels, n_records),
    'units_sold': np.random.randint(1, 50, n_records),
    'unit_price': np.random.uniform(10, 1000, n_records).round(2),
    'discount': np.random.choice([0, 0.05, 0.1, 0.15, 0.2], n_records)
})

# Calculate total sales
sales_data['total_sale'] = sales_data['units_sold'] * sales_data['unit_price'] * (1 - sales_data['discount'])

print("Generated sales dataset:")
display(sales_data.head())

# Save the data in different formats
sales_csv_path = sample_dir / "sales_data.csv"
sales_json_path = sample_dir / "sales_data.json"
sales_pkl_path = sample_dir / "sales_data.pkl"

sales_data.to_csv(sales_csv_path, index=False)
sales_data.to_json(sales_json_path, orient='records')
sales_data.to_pickle(sales_pkl_path)

print(f"Saved sales data in three formats: CSV, JSON, and Pickle")

# Perform analysis and aggregation
def analyze_sales(data_path, format='csv'):
    """Analyze sales data from a file"""
    if format == 'csv':
        data = pd.read_csv(data_path)
        if 'date' in data.columns:
            data['date'] = pd.to_datetime(data['date'])
    elif format == 'json':
        data = pd.read_json(data_path, orient='records')
    elif format == 'pickle':
        data = pd.read_pickle(data_path)
    else:
        raise ValueError(f"Unsupported format: {format}")
    
    # Monthly sales trend
    monthly_sales = data.groupby(data['date'].dt.strftime('%Y-%m'))[['total_sale']].sum()
    
    # Product performance
    product_sales = data.groupby('product').agg({
        'units_sold': 'sum',
        'total_sale': 'sum'
    }).sort_values('total_sale', ascending=False)
    
    # Regional performance
    region_sales = data.groupby('region')[['total_sale']].sum().sort_values('total_sale', ascending=False)
    
    # Channel analysis
    channel_sales = data.groupby('channel')[['total_sale']].sum().sort_values('total_sale', ascending=False)
    
    return {
        'monthly_trend': monthly_sales,
        'product_performance': product_sales,
        'regional_performance': region_sales,
        'channel_performance': channel_sales
    }

# Run the analysis on our CSV file
analysis_results = analyze_sales(sales_csv_path)

print("\nMonthly Sales Trend:")
display(analysis_results['monthly_trend'].head())

print("\nProduct Performance:")
display(analysis_results['product_performance'])

print("\nRegional Performance:")
display(analysis_results['regional_performance'])

print("\nChannel Performance:")
display(analysis_results['channel_performance'])

# Save the analysis results as JSON for reporting
analysis_json_path = sample_dir / "sales_analysis.json"

# We need to convert DataFrames to serializable format
def prepare_for_json(data_dict):
    result = {}
    for k, v in data_dict.items():
        if isinstance(v, pd.DataFrame):
            result[k] = v.reset_index().to_dict(orient='records')
        else:
            result[k] = v
    return result

with open(analysis_json_path, 'w') as f:
    json.dump(prepare_for_json(analysis_results), f, indent=2, default=str)

print(f"\nSaved analysis results to {analysis_json_path}")

### 7.4 Saving and Loading Machine Learning Models

Here, we'll train a machine learning model and save it using pickle:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Prepare data for ML model
X = sales_data[['units_sold', 'unit_price', 'discount']]
y = sales_data['total_sale']

# Add categorical features (after encoding)
X = pd.concat([X, pd.get_dummies(sales_data[['product', 'region', 'channel']])], axis=1)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

# Train a model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)

print("\nModel performance:")
print(f"Training R² score: {r2_score(y_train, train_preds):.4f}")
print(f"Test R² score: {r2_score(y_test, test_preds):.4f}")
print(f"Test MAE: ${mean_absolute_error(y_test, test_preds):.2f}")

# Save the model using pickle
model_path = sample_dir / "sales_prediction_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"\nSaved model to {model_path}")

# Later, we can load the model and use it for predictions
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Create a sample for prediction
sample = X_test.iloc[0:1]  # Just use the first test sample

# Make prediction
prediction = loaded_model.predict(sample)[0]
actual = y_test.iloc[0]

print("\nPrediction from loaded model:")
print(f"Predicted sale: ${prediction:.2f}")
print(f"Actual sale: ${actual:.2f}")
print(f"Error: ${abs(prediction - actual):.2f}")

# Save model metadata in JSON format
model_meta = {
    "model_type": "RandomForestRegressor",
    "trained_on": str(datetime.now()),
    "parameters": {
        "n_estimators": 100,
        "random_state": 42
    },
    "performance": {
        "train_r2": float(r2_score(y_train, train_preds)),
        "test_r2": float(r2_score(y_test, test_preds)),
        "test_mae": float(mean_absolute_error(y_test, test_preds))
    },
    "feature_importance": {
        feature: float(importance) 
        for feature, importance in zip(X.columns, model.feature_importances_)
    }
}

meta_path = sample_dir / "model_metadata.json"
with open(meta_path, 'w') as f:
    json.dump(model_meta, f, indent=2)

print(f"\nSaved model metadata to {meta_path}")

## Summary and Best Practices

In this notebook, we've explored three key file formats in Python:

1. **CSV (Comma-Separated Values)**
   - Best for tabular data
   - Human-readable and widely supported
   - Limited to flat data structures
   - Common tools: `csv` module and `pandas`

2. **JSON (JavaScript Object Notation)**
   - Great for hierarchical/nested data
   - Human-readable and widely used for web APIs
   - Preserves common data types
   - Common tools: `json` module and `pandas`

3. **Pickle**
   - Python-specific binary format
   - Preserves almost any Python object
   - Not human-readable or cross-language compatible
   - Security concerns - never unpickle untrusted data
   - Common tools: `pickle` module

### Best Practices:

1. **Always use context managers (`with` statements) for file operations**
2. **Implement proper error handling for file operations**
3. **Choose the right format for your needs:**
   - Use CSV for simple tabular data
   - Use JSON for structured data, especially when sharing with other systems
   - Use Pickle only for Python-specific objects in trusted environments
4. **Be mindful of security implications, especially with Pickle**
5. **Use pandas when possible for convenient data manipulation**
6. **Document your file formats and include metadata when appropriate**

By understanding these file formats and their appropriate use cases, you can make your Python data handling more efficient and robust.